# LangChain Hands-on 06 — Document Loaders
**Trainer:** Ajeetkumar

**What we will practise**
- What is a `Document`?
- TextLoader, CSVLoader, JSONLoader
- PyPDFLoader (PDF)
- WebBaseLoader (website)
- WikipediaLoader, YouTube transcript idea
- DirectoryLoader (many files)
- lazy loading for big data

> Run on Google Colab. Add your Groq key in Colab **Secrets** (🔑 icon) with the name `groq_key` (free key: console.groq.com).

In [ ]:
! pip install -q langchain langchain-community pypdf beautifulsoup4 wikipedia jq

# What is a Document Loader?
- Loads data from a source (PDF, website, CSV, DB …) into LangChain **Document** objects.
- `Document` = **page_content** (text) + **metadata** (source, page number, …)
- First step of every **RAG** app.

```
PDF / Web / CSV ──▶ Loader ──▶ [Document(page_content, metadata), ...]
```

In [ ]:
from langchain_core.documents import Document

In [ ]:
doc = Document(page_content="LangChain helps build LLM apps.", metadata={"source": "my_notes", "author": "Ajeet"})
doc

In [ ]:
doc.page_content, doc.metadata

## Create some sample files

In [ ]:
open("bank_policy.txt", "w").write("""NovaBank Card Policy
Report a lost card within 3 days for zero liability.
Card-not-present transactions above Rs 5,000 need OTP.
International usage is disabled by default.""")

open("students.csv", "w").write("""name,course,city
Ajeet,GenAI,Pune
Priya,Data Science,Mumbai
Rahul,Python,Nagpur""")

import json
json.dump([{"id": 1, "question": "What is AI?", "answer": "Machines that think."},
           {"id": 2, "question": "What is ML?", "answer": "Learning from data."}], open("faq.json", "w"))

## 1. TextLoader

In [ ]:
from langchain_community.document_loaders import TextLoader

In [ ]:
docs = TextLoader("bank_policy.txt").load()
docs

In [ ]:
print(docs[0].page_content)
print(docs[0].metadata)

## 2. CSVLoader — one Document per row

In [ ]:
from langchain_community.document_loaders import CSVLoader

In [ ]:
docs = CSVLoader("students.csv").load()
len(docs)

In [ ]:
for d in docs:
    print(d.page_content, "|", d.metadata)

## 3. JSONLoader — pick fields with a jq query

In [ ]:
from langchain_community.document_loaders import JSONLoader

In [ ]:
docs = JSONLoader("faq.json", jq_schema=".[].answer", text_content=False).load()
docs

## 4. PyPDFLoader — one Document per page

In [ ]:
! wget -q -O attention.pdf https://arxiv.org/pdf/1706.03762

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
pages = PyPDFLoader("attention.pdf").load()
len(pages)

In [ ]:
print(pages[0].page_content[:500])

In [ ]:
pages[0].metadata

## 5. WebBaseLoader — load a web page

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

In [ ]:
web_docs = WebBaseLoader("https://en.wikipedia.org/wiki/LangChain").load()
print(web_docs[0].metadata)
print(web_docs[0].page_content[:500])

## 6. WikipediaLoader

In [ ]:
from langchain_community.document_loaders import WikipediaLoader

In [ ]:
wiki_docs = WikipediaLoader(query="Retrieval-augmented generation", load_max_docs=2).load()
for d in wiki_docs:
    print(d.metadata["title"], "→", d.page_content[:150], "\n")

## 7. DirectoryLoader — load all files from a folder

In [ ]:
import os, shutil
os.makedirs("kb", exist_ok=True)
for name, text in {"upi.txt": "UPI limit for new payee is Rs 25,000 in first 24 hours.",
                   "kyc.txt": "Video KYC is available 9 AM to 9 PM."}.items():
    open(f"kb/{name}", "w").write(text)

In [ ]:
from langchain_community.document_loaders import DirectoryLoader

In [ ]:
docs = DirectoryLoader("kb", glob="*.txt", loader_cls=TextLoader).load()
for d in docs:
    print(d.metadata["source"], "→", d.page_content)

## 8. lazy_load — one document at a time (for very big data)

In [ ]:
for page in PyPDFLoader("attention.pdf").lazy_load():
    print(page.metadata["page"], len(page.page_content))
    if page.metadata["page"] >= 2:
        break

## 9. Use loaded documents with the LLM

In [ ]:
! pip install -q langchain-groq
from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('groq_key')
from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b")

In [ ]:
text = pages[0].page_content[:3000]
print(llm.invoke(f"Summarise this research paper page in 5 bullet points:\n{text}").content)

# Popular loaders (cheat sheet)
| Source | Loader | Package |
|---|---|---|
| .txt | `TextLoader` | community |
| .pdf | `PyPDFLoader`, `PyMuPDFLoader` | pypdf / pymupdf |
| .csv | `CSVLoader` | community |
| .docx | `Docx2txtLoader` | docx2txt |
| website | `WebBaseLoader`, `RecursiveUrlLoader` | bs4 |
| Wikipedia | `WikipediaLoader` | wikipedia |
| YouTube | `YoutubeLoader` | youtube-transcript-api |
| folder | `DirectoryLoader` | community |
| any file | `UnstructuredFileLoader` | unstructured |

## 🧪 Practice
1. Load your own resume PDF and ask the LLM for 5 interview questions.
2. Load 3 Wikipedia pages on Indian cities and print their titles.
3. Load a website and count the words.